In [1]:
# CELL 1 STUDENT CONFIGURATION
# Path is used for file and folder paths
from pathlib import Path
 
# Student-specific information (REPLACE the ID with your own)
LAST_NAME = "Endaya"
STUDENT_ID = "TUPM-26-4330"
 
# Seed digit = last digit of the student ID
SEED_DIGIT = int(STUDENT_ID[-1])
# ID sum = sum of every digit in the student ID
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
 
# Display the derived values
print("Student:", LAST_NAME, STUDENT_ID)
print("Seed digit:", SEED_DIGIT)
print("ID sum:", ID_SUM)

Student: Endaya TUPM-26-4330
Seed digit: 0
ID sum: 18


In [2]:
# CELL 2 CREATE WORKSPACE FOLDERS
# Define the folders (put the notebook inside Lab6_File_Handling_Additional_Exercises)
WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"        # raw Kaggle CSV goes here (NOT uploaded to GitHub)
OUTPUT_DIR = WORK_DIR / "output"    # generated files
BACKUP_DIR = WORK_DIR / "backup"    # backups
 
# Create the folders if they do not exist yet
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)
 
# Verify the workspace
print("Workspace:", WORK_DIR.resolve())
print("Data folder exists:", DATA_DIR.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Backup folder exists:", BACKUP_DIR.exists())

Workspace: C:\Users\WINDOWS\Lab6_File_Handling_Additional_Exercise
Data folder exists: True
Output folder exists: True
Backup folder exists: True


In [3]:
# CELL 3 EXERCISE 3 PARAMETERS
# Built-in modules only (no Pandas or NumPy)
import csv
import json
import shutil
from collections import Counter
 
# Dataset location (download Global_Cybersecurity_Threats_2015-2024.csv from Kaggle)
dataset_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
 
# Student-specific rules
target_year = 2015 + SEED_DIGIT
loss_field = "Financial Loss (in Million $)"
count_field = "Attack Type"
 
print("Target year:", target_year)
print("Loss field:", loss_field)
print("Count field:", count_field)

Target year: 2015
Loss field: Financial Loss (in Million $)
Count field: Attack Type


In [4]:
# CELL 4 READ AND SELECT INCIDENTS FOR THE TARGET YEAR
total_records = 0        # all data rows
invalid_loss = 0         # target-year rows with a missing/invalid loss
selected = []            # (row, loss) pairs in original dataset order
fieldnames = []          # original CSV column names
attack_counts = Counter()
total_loss = 0.0
 
if not dataset_path.exists():
    print("Dataset not found:", dataset_path)
else:
    with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        fieldnames = reader.fieldnames
 
        for row in reader:
            total_records += 1
 
            # Keep only rows from the target year (skip rows with a bad year)
            try:
                year = int(row["Year"])
            except (KeyError, TypeError, ValueError):
                continue
            if year != target_year:
                continue
 
            # Read the financial loss; handle invalid values
            try:
                loss = float(row[loss_field])
            except (KeyError, TypeError, ValueError):
                invalid_loss += 1
                continue
 
            selected.append((row, loss))
            attack_counts[row[count_field]] += 1
            total_loss += loss
 
# Average loss (0 when there are no incidents)
average_loss = total_loss / len(selected) if selected else 0.0
 
print("Total records:", total_records)
print("Incidents in target year:", len(selected))
print("Invalid loss values skipped:", invalid_loss)
print("Count per attack type:", dict(attack_counts))
print("Total loss (USD millions):", round(total_loss, 2))
print("Average loss (USD millions):", round(average_loss, 2))

Dataset not found: data\Global_Cybersecurity_Threats_2015-2024.csv
Total records: 0
Incidents in target year: 0
Invalid loss values skipped: 0
Count per attack type: {}
Total loss (USD millions): 0.0
Average loss (USD millions): 0.0


In [7]:
# CELL 5 FIND THE FIVE COSTLIEST INCIDENTS
# Sort by loss, highest first. sorted() is stable, so tied losses keep their dataset order.
ranked = sorted(selected, key=lambda pair: -pair[1])[:5]
 
# Build the list that will be exported to JSON
top_five = []
for rank, (row, loss) in enumerate(ranked, 1):
    record = dict(row)
    record["Rank"] = rank
    top_five.append(record)
 
print("Five costliest incidents:")
for record in top_five:
    print(" ", record["Rank"], record["Country"], record["Attack Type"], record[loss_field])

Five costliest incidents:


In [8]:
# CELL 6 EXPORT FILTERED CSV AND JSON
# Filtered CSV: all selected incidents with the original columns
filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incidents_{target_year}.csv"
with open(filtered_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames or [])
    writer.writeheader()
    writer.writerows(row for row, _ in selected)
 
# JSON file: summary and the five costliest incidents
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incident_summary.json"
summary = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "target_year": target_year,
    "total_records": total_records,
    "incidents_in_target_year": len(selected),
    "count_per_attack_type": dict(attack_counts),
    "total_financial_loss_million_usd": round(total_loss, 2),
    "average_financial_loss_million_usd": round(average_loss, 2),
    "five_costliest_incidents": top_five,
}
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)
 
print("Created:", filtered_path)
print("Created:", json_path)

Created: output\Endaya_TUPM-26-4330_incidents_2015.csv
Created: output\Endaya_TUPM-26-4330_incident_summary.json


In [9]:
# CELL 7 BACKUP AND VERIFY THE JSON FILE
# Copy the JSON file
backup_path = BACKUP_DIR / json_path.name
shutil.copy2(json_path, backup_path)
 
# Verify: the backup exists and its content is identical to the original
with open(json_path, "r", encoding="utf-8") as original, open(backup_path, "r", encoding="utf-8") as backup:
    identical = original.read() == backup.read()
backup_status = "VERIFIED" if backup_path.exists() and identical else "FAILED"
 
print("Backup:", backup_path)
print("Backup verification:", backup_status)

Backup: backup\Endaya_TUPM-26-4330_incident_summary.json
Backup verification: VERIFIED


In [11]:
# CELL 8 GENERATE THE TEXT REPORT
# Path of the report
report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incident_report.txt"
 
with open(report_path, "w", encoding="utf-8") as report:
    report.write("CYBERSECURITY INCIDENT REPORT\n")
    report.write("-" * 40 + "\n")
    report.write(f"Student: {LAST_NAME} ({STUDENT_ID})\n")
    report.write(f"Target Year: {target_year}\n")
    report.write(f"Total Records in Dataset: {total_records}\n")
    report.write(f"Incidents in Target Year: {len(selected)}\n\n")
 
    # Incident counts per attack type
    report.write("Incident Count per Attack Type:\n")
    for attack, count in attack_counts.items():
        report.write(f"  {attack}: {count}\n")
 
    # Financial losses
    report.write(f"\nTotal Financial Loss (USD millions): {total_loss:.2f}\n")
    report.write(f"Average Financial Loss (USD millions): {average_loss:.2f}\n\n")
 
    # Five costliest incidents
    report.write("Five Costliest Incidents:\n")
    for record in top_five:
        report.write(
            f"  {record['Rank']}. {record['Country']} | {record[count_field]} | "
            f"{record[loss_field]} million USD\n"
        )
 
    # Output file information
    report.write("\nOutput Files:\n")
    report.write(f"  Filtered CSV: {filtered_path.name}\n")
    report.write(f"  JSON File: {json_path.name}\n")
    report.write(f"  Backup File: {backup_path.name} ({backup_status})\n")
 
print("Report created:", report_path)

Report created: output\Endaya_TUPM-26-4330_incident_report.txt


In [12]:
# CELL 9 FINAL OUTPUT (ASSESSMENT DATA)
print("Target year:", target_year)
print("Total records in dataset:", total_records)
print("Incidents in target year:", len(selected))
print("Count per attack type:", dict(attack_counts))
print("Total financial loss (USD millions):", round(total_loss, 2))
print("Average financial loss (USD millions):", round(average_loss, 2))
print("Five costliest incidents:", [(r["Rank"], r["Country"], r[loss_field]) for r in top_five])
print("Filtered CSV file:", filtered_path.name)
print("JSON summary file:", json_path.name)
print("Backup verification:", backup_status)
print("Report file:", report_path.name)

Target year: 2015
Total records in dataset: 0
Incidents in target year: 0
Count per attack type: {}
Total financial loss (USD millions): 0.0
Average financial loss (USD millions): 0.0
Five costliest incidents: []
Filtered CSV file: Endaya_TUPM-26-4330_incidents_2015.csv
JSON summary file: Endaya_TUPM-26-4330_incident_summary.json
Backup verification: VERIFIED
Report file: Endaya_TUPM-26-4330_incident_report.txt
